In [1]:
import pandas as pd
import numpy as np

In [2]:
batting_path = r"C:\stuff\DATA MINING PROJECT\data\processed\ipl\batting_match.csv"
bowling_path = r"C:\stuff\DATA MINING PROJECT\data\processed\ipl\bowling_match.csv"

batting = pd.read_csv(batting_path)
bowling = pd.read_csv(bowling_path)

print("Batting shape:", batting.shape)
print("Bowling shape:", bowling.shape)

Batting shape: (18768, 11)
Bowling shape: (14700, 13)


In [ ]:
recent_seasons = ["2022", "2023", "2024", "2025", "2026"]

batting = batting[
    batting["season"].astype(str).isin(recent_seasons)
].copy()

bowling = bowling[
    bowling["season"].astype(str).isin(recent_seasons)
].copy()

print("Batting shape:", batting.shape)
print("Bowling shape:", bowling.shape)

print("\nSeasons:")
print(sorted(batting["season"].unique()))

Batting shape: (5697, 11)
Bowling shape: (4334, 13)

Seasons:
['2022', '2023', '2024', '2025', '2026']


In [4]:
print("Batting records by season:")
display(
    batting.groupby("season")
    .size()
    .reset_index(name="records")
)

print("\nBowling records by season:")
display(
    bowling.groupby("season")
    .size()
    .reset_index(name="records")
)

Batting records by season:


,season,records
0,2022,1158
1,2023,1171
2,2024,1115
3,2025,1123
4,2026,1130



Bowling records by season:


,season,records
0,2022,857
1,2023,898
2,2024,857
3,2025,868
4,2026,854


In [5]:
batting_season = (
    batting
    .groupby(["player", "season"], as_index=False)
    .agg(
        batting_matches=("match_id", "nunique"),
        runs=("runs", "sum"),
        batting_balls=("balls", "sum"),
        fours=("fours", "sum"),
        sixes=("sixes", "sum")
    )
)

# Strike rate
batting_season["strike_rate"] = (
    batting_season["runs"] /
    batting_season["batting_balls"].replace(0, np.nan)
) * 100

# Boundary runs
batting_season["boundary_runs"] = (
    batting_season["fours"] * 4 +
    batting_season["sixes"] * 6
)

# Percentage of runs from boundaries
batting_season["boundary_percentage"] = (
    batting_season["boundary_runs"] /
    batting_season["runs"].replace(0, np.nan)
) * 100

print("Shape:", batting_season.shape)

display(batting_season.head(20))

Shape: (864, 10)


,player,season,batting_matches,runs,batting_balls,fours,sixes,strike_rate,boundary_runs,boundary_percentage
0,A Badoni,2022,11,161,130,11,7,123.846154,86,53.416149
1,A Badoni,2023,12,238,171,13,12,139.181287,124,52.100840
2,A Badoni,2024,12,235,171,22,5,137.426901,118,50.212766
3,A Badoni,2025,11,329,221,27,14,148.868778,192,58.358663
4,A Badoni,2026,10,215,141,23,9,152.482270,146,67.906977
5,A Kamboj,2024,1,2,2,0,0,100.000000,0,0.000000
6,A Kamboj,2025,5,14,12,2,0,116.666667,8,57.142857
7,A Kamboj,2026,4,58,36,3,4,161.111111,36,62.068966
8,A Manohar,2022,7,108,75,14,3,144.000000,74,68.518519
9,A Manohar,2023,6,114,83,6,7,137.349398,66,57.894737


In [6]:
print("Unique batting players:", batting_season["player"].nunique())
print("Seasons:", batting_season["season"].nunique())

print("\nMissing values:")
display(batting_season.isna().sum())

Unique batting players: 343
Seasons: 5

Missing values:


player                  0
season                  0
batting_matches         0
runs                    0
batting_balls           0
fours                   0
sixes                   0
strike_rate             0
boundary_runs           0
boundary_percentage    44
dtype: int64

In [7]:
# ============================================
# BOWLING — PLAYER-SEASON FEATURES
# ============================================

bowling_season = (
    bowling
    .groupby(["player", "season"], as_index=False)
    .agg(
        bowling_matches=("match_id", "nunique"),
        bowling_balls=("balls", "sum"),
        runs_conceded=("runs_conceded", "sum"),
        wickets=("wickets", "sum")
    )
)

# Overs
bowling_season["overs"] = (
    bowling_season["bowling_balls"] / 6
)

# Economy
bowling_season["economy"] = (
    bowling_season["runs_conceded"] /
    bowling_season["overs"].replace(0, np.nan)
)

# Bowling strike rate
bowling_season["bowling_strike_rate"] = (
    bowling_season["bowling_balls"] /
    bowling_season["wickets"].replace(0, np.nan)
)

print("Shape:", bowling_season.shape)

display(bowling_season.head(20))

Shape: (653, 9)


,player,season,bowling_matches,bowling_balls,runs_conceded,wickets,overs,economy,bowling_strike_rate
0,A Badoni,2022,2,12,11,2,2.000000,5.500000,6.000000
1,A Badoni,2023,1,1,0,0,0.166667,0.000000,NaN
2,A Badoni,2024,2,12,25,0,2.000000,12.500000,NaN
3,A Badoni,2025,2,10,13,2,1.666667,7.800000,5.000000
4,A Badoni,2026,1,6,14,0,1.000000,14.000000,NaN
5,A Kamboj,2024,3,60,114,2,10.000000,11.400000,30.000000
6,A Kamboj,2025,8,129,172,8,21.500000,8.000000,16.125000
7,A Kamboj,2026,14,302,530,21,50.333333,10.529801,14.380952
8,A Mishra,2023,7,114,149,7,19.000000,7.842105,16.285714
9,A Mishra,2024,1,12,20,1,2.000000,10.000000,12.000000


In [8]:
print("Unique bowling players:", bowling_season["player"].nunique())
print("Seasons:", bowling_season["season"].nunique())

print("\nMissing values:")
display(bowling_season.isna().sum())

Unique bowling players: 262
Seasons: 5

Missing values:


player                   0
season                   0
bowling_matches          0
bowling_balls            0
runs_conceded            0
wickets                  0
overs                    0
economy                  0
bowling_strike_rate    119
dtype: int64

In [9]:
# ============================================
# COMBINE BATTING + BOWLING
# ============================================

player_season = pd.merge(
    batting_season,
    bowling_season,
    on=["player", "season"],
    how="outer"
)

print("Player-season shape:", player_season.shape)
print("Unique players:", player_season["player"].nunique())
print("Seasons:", player_season["season"].nunique())

display(player_season.head(20))

Player-season shape: (1010, 17)
Unique players: 384
Seasons: 5


,player,season,batting_matches,runs,batting_balls,fours,sixes,strike_rate,boundary_runs,boundary_percentage,bowling_matches,bowling_balls,runs_conceded,wickets,overs,economy,bowling_strike_rate
0,A Badoni,2022,11.0,161.0,130.0,11.0,7.0,123.846154,86.0,53.416149,2.0,12.0,11.0,2.0,2.000000,5.500000,6.000000
1,A Badoni,2023,12.0,238.0,171.0,13.0,12.0,139.181287,124.0,52.100840,1.0,1.0,0.0,0.0,0.166667,0.000000,NaN
2,A Badoni,2024,12.0,235.0,171.0,22.0,5.0,137.426901,118.0,50.212766,2.0,12.0,25.0,0.0,2.000000,12.500000,NaN
3,A Badoni,2025,11.0,329.0,221.0,27.0,14.0,148.868778,192.0,58.358663,2.0,10.0,13.0,2.0,1.666667,7.800000,5.000000
4,A Badoni,2026,10.0,215.0,141.0,23.0,9.0,152.482270,146.0,67.906977,1.0,6.0,14.0,0.0,1.000000,14.000000,NaN
5,A Kamboj,2024,1.0,2.0,2.0,0.0,0.0,100.000000,0.0,0.000000,3.0,60.0,114.0,2.0,10.000000,11.400000,30.000000
6,A Kamboj,2025,5.0,14.0,12.0,2.0,0.0,116.666667,8.0,57.142857,8.0,129.0,172.0,8.0,21.500000,8.000000,16.125000
7,A Kamboj,2026,4.0,58.0,36.0,3.0,4.0,161.111111,36.0,62.068966,14.0,302.0,530.0,21.0,50.333333,10.529801,14.380952
8,A Manohar,2022,7.0,108.0,75.0,14.0,3.0,144.000000,74.0,68.518519,NaN,NaN,NaN,NaN,NaN,NaN,NaN
9,A Manohar,2023,6.0,114.0,83.0,6.0,7.0,137.349398,66.0,57.894737,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [10]:
count_columns = [
    "batting_matches",
    "runs",
    "batting_balls",
    "fours",
    "sixes",
    "boundary_runs",
    "bowling_matches",
    "bowling_balls",
    "runs_conceded",
    "wickets",
    "overs"
]

for col in count_columns:
    player_season[col] = player_season[col].fillna(0)

In [11]:
print("Final player-season shape:", player_season.shape)

print(
    "Unique players:",
    player_season["player"].nunique()
)

print(
    "Seasons:",
    sorted(player_season["season"].unique())
)

display(
    player_season
    .sort_values(["player", "season"])
    .head(30)
)

Final player-season shape: (1010, 17)
Unique players: 384
Seasons: ['2022', '2023', '2024', '2025', '2026']


,player,season,batting_matches,runs,batting_balls,fours,sixes,strike_rate,boundary_runs,boundary_percentage,bowling_matches,bowling_balls,runs_conceded,wickets,overs,economy,bowling_strike_rate
0,A Badoni,2022,11.0,161.0,130.0,11.0,7.0,123.846154,86.0,53.416149,2.0,12.0,11.0,2.0,2.000000,5.500000,6.000000
1,A Badoni,2023,12.0,238.0,171.0,13.0,12.0,139.181287,124.0,52.100840,1.0,1.0,0.0,0.0,0.166667,0.000000,NaN
2,A Badoni,2024,12.0,235.0,171.0,22.0,5.0,137.426901,118.0,50.212766,2.0,12.0,25.0,0.0,2.000000,12.500000,NaN
3,A Badoni,2025,11.0,329.0,221.0,27.0,14.0,148.868778,192.0,58.358663,2.0,10.0,13.0,2.0,1.666667,7.800000,5.000000
4,A Badoni,2026,10.0,215.0,141.0,23.0,9.0,152.482270,146.0,67.906977,1.0,6.0,14.0,0.0,1.000000,14.000000,NaN
5,A Kamboj,2024,1.0,2.0,2.0,0.0,0.0,100.000000,0.0,0.000000,3.0,60.0,114.0,2.0,10.000000,11.400000,30.000000
6,A Kamboj,2025,5.0,14.0,12.0,2.0,0.0,116.666667,8.0,57.142857,8.0,129.0,172.0,8.0,21.500000,8.000000,16.125000
7,A Kamboj,2026,4.0,58.0,36.0,3.0,4.0,161.111111,36.0,62.068966,14.0,302.0,530.0,21.0,50.333333,10.529801,14.380952
8,A Manohar,2022,7.0,108.0,75.0,14.0,3.0,144.000000,74.0,68.518519,0.0,0.0,0.0,0.0,0.000000,NaN,NaN
9,A Manohar,2023,6.0,114.0,83.0,6.0,7.0,137.349398,66.0,57.894737,0.0,0.0,0.0,0.0,0.000000,NaN,NaN


In [12]:
# ============================================
# SAVE PLAYER-SEASON FEATURES
# ============================================

output_path = r"C:\stuff\DATA MINING PROJECT\data\processed\ipl\player_season_features.csv"

player_season.to_csv(
    output_path,
    index=False
)

print("Saved successfully!")
print(output_path)

Saved successfully!
C:\stuff\DATA MINING PROJECT\data\processed\ipl\player_season_features.csv
